# Task 2.1 — Dataset Selection and Setup
**Paper:** Efficient Additive Kernels via Explicit Feature Maps  
**Student:** Nikhil Raj | Roll No: 230080

---

## Random Seed Declaration
All random seeds used in this notebook are set at the top. `random_state=42` is used throughout for reproducibility.

---

## Dataset Choice: `sklearn.datasets.load_digits`

**What the dataset is:**  
The Digits dataset (available natively in scikit-learn) contains 1,797 grayscale images of handwritten digits (0–9), each represented as a flattened 8×8 pixel grid, giving 64 features per sample. Feature values are pixel intensities in the range [0, 16] — integers, always non-negative.

**Why it is a reasonable testbed for this paper's method:**  
The homogeneous kernel map is specifically designed for histogram-type features — vectors of non-negative counts or frequencies. Pixel intensity bins are structurally identical to visual word histograms (like the bag-of-visual-words vectors used in the paper's Caltech-101 experiments): both are non-negative integer-valued vectors where each entry counts how much of something is present. The chi-squared kernel, which the paper uses as its primary kernel, is a standard measure of dissimilarity between two histograms — making it semantically meaningful for pixel intensity distributions. The dataset is also natively built into scikit-learn, requiring zero download, which directly satisfies the exam's compute feasibility requirement.

**Limitations compared to the paper's original dataset:**  
The paper's primary benchmark, Caltech-101, uses 4,200-dimensional multi-scale dense SIFT histograms with spatial pyramid pooling across 102 object categories. Load_digits is far simpler: 64-dimensional raw pixel values, 10 digit classes, no spatial structure beyond the implicit 8×8 grid, and no learnt visual vocabulary. The features in load_digits are not truly semantic histogram bins — they are raw pixel intensities, which means the χ² kernel's statistical interpretation (measuring histogram divergence) is weaker than it would be for bag-of-words or HOG features. As a result, the absolute accuracy values will differ from the paper's reported numbers, but the *relative* comparison between linear SVM and the approximate kernel SVM should still demonstrate the same directional improvement the paper claims.

**Preprocessing:**  
MinMaxScaler is applied to scale all features to [0, 1]. This preserves non-negativity (the critical requirement of the feature map) while normalising the scale across features so that no single pixel dominates the chi-squared computation.


In [1]:
# ── RANDOM SEED DECLARATION ──────────────────────────────────────────────────
RANDOM_STATE = 42

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

np.random.seed(RANDOM_STATE)

# ── LOAD DATASET ──────────────────────────────────────────────────────────────
X, y = load_digits(return_X_y=True)

print(f"Dataset: load_digits")
print(f"Total samples      : {X.shape[0]}")
print(f"Features per sample: {X.shape[1]}")
print(f"Classes            : {np.unique(y)}")
print(f"Feature range BEFORE scaling: [{X.min():.1f}, {X.max():.1f}]")
print(f"Any negative values: {(X < 0).any()}")


Dataset: load_digits
Total samples      : 1797
Features per sample: 64
Classes            : [0 1 2 3 4 5 6 7 8 9]
Feature range BEFORE scaling: [0.0, 16.0]
Any negative values: False


**Code explanation:** This cell loads the raw digit dataset and confirms the two most important properties: (1) 64 non-negative features per sample (pixel intensities 0–16), and (2) 10 classes. The non-negativity check is critical because the feature map (Eq. 19) requires x ≥ 0 — a domain assumption explicitly stated in Section 2 of the paper.

In [2]:
# ── PREPROCESSING: MinMaxScaler → [0, 1] ─────────────────────────────────────
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

print(f"Feature range AFTER scaling : [{X_scaled.min():.4f}, {X_scaled.max():.4f}]")
print(f"Any negative values after scaling: {(X_scaled < 0).any()}")
print(f"Any zero values: {(X_scaled == 0).any()} — handled by AdditiveChi2Sampler internally")

# ── TRAIN / TEST SPLIT ────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
print(f"\nTrain samples: {X_train.shape[0]}")
print(f"Test  samples: {X_test.shape[0]}")


Feature range AFTER scaling : [0.0000, 1.0000]
Any negative values after scaling: False
Any zero values: True — handled by AdditiveChi2Sampler internally

Train samples: 1347
Test  samples: 450


**Code explanation:** MinMaxScaler maps all features to [0, 1], preserving non-negativity. This corresponds to the normalisation requirement discussed in Section 2.2 of the paper: "the normalisation condition K(**x**,**x**) = const. can be enforced by scaling the histograms." A 75/25 stratified train/test split is used to ensure class balance.

In [3]:
# ── VISUALISE SAMPLE IMAGES ──────────────────────────────────────────────────
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(X[i].reshape(8, 8), cmap='gray')
    ax.set_title(f'Label: {y[i]}')
    ax.axis('off')
plt.suptitle('Sample images from load_digits dataset', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('results/dataset_samples.png', dpi=120, bbox_inches='tight')
plt.close()
print("Saved: results/dataset_samples.png")
print("Each image is an 8x8 pixel grid — flattened to 64 non-negative integer features.")


Saved: results/dataset_samples.png
Each image is an 8x8 pixel grid — flattened to 64 non-negative integer features.


**Code explanation:** Visualises 10 sample images from the dataset as 8×8 grids. This confirms the histogram-like nature of the features: each pixel's intensity value is a local count of ink density, structurally analogous to the HOG and SIFT histogram bins used in the paper's experiments (Section 8).